# Lesson 11 — GraphRAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/11-graphrag/colab.ipynb)

*Some questions are about connections, not resemblance. But a graph only helps if the corpus has one.*

**What you will learn**

- What questions vector search structurally cannot answer, and how a graph helps
- Entity extraction, co-occurrence edges, and traversal retrieval — step by step
- How to tell, in thirty seconds, whether your corpus has enough structure (mean degree)
- Why GraphRAG scored 0.000 on vocabulary-mismatch questions here
- Two silent extraction bugs, and building a small corpus where the graph *does* work

| | |
|---|---|
| ⏱️ Time | 45 min |
| 🖥️ Runtime | CPU is fine |
| 🤖 Needs the generator (Ollama) | no — retrieval only |

**Before this:** [10-agentic-rag](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/10-agentic-rag/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = False  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. The idea

Vector search retrieves passages that *resemble* the query. Some questions are not
about resemblance at all:

- "How do X and Y relate?"
- "What else is affected by Z?"
- "What depends on this component?"

These need the **connections between** passages, and an embedding of a passage does not
encode what it connects to. GraphRAG builds that structure explicitly:

```
nodes = entities (and the chunks they appear in)
edges = two entities co-occurring in the same chunk, weighted by how often
retrieval = find entities named in the query, walk to neighbours, return their chunks
```

Co-occurrence is a weak notion of "related" — it records that two things were
discussed together, not *how*. A full GraphRAG extracts typed relations ("X depends on
Y") with a model call per chunk, at build time and again on every corpus change.

## 2. Step 1 — extract entities

`ragkit`'s heuristic extractor needs no model: it matches things that are
*typographically marked* as entities in technical writing.

In [ ]:
from ragkit.config import get_settings
from ragkit.graph.extract import ENTITY_PATTERNS, extract_heuristic
from ragkit.ingest import RecursiveChunker, chunk_documents, load_documents

for p in ENTITY_PATTERNS:
    print(p.pattern)

settings = get_settings()
chunks = chunk_documents(
    load_documents(), RecursiveChunker(settings.chunk_size, settings.chunk_overlap)
)
sample = next(c for c in chunks if c.doc_id == "07-operations.md" and "FUSION_CONSTANT" in c.text)
print("\nchunk:", sample.text[:300].replace("\n", " "), "…")
print("entities:", extract_heuristic(sample))

## 3. Step 2 — build the graph, and look at it

In [ ]:
from ragkit.graph import GraphRetriever, build_graph

graph = build_graph(chunks)
graph.stats()

In [ ]:
!{sys.executable} lessons/11-graphrag/run.py

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.Graph()
for a, neighbours in graph.edges.items():
    for b, w in neighbours.items():
        G.add_edge(graph.display.get(a, a), graph.display.get(b, b), weight=w)
isolated = graph.n_entities - G.number_of_nodes()

plt.figure(figsize=(8, 4))
nx.draw_networkx(
    G,
    nx.spring_layout(G, seed=1),
    node_color="#4C72B0",
    font_size=9,
    font_color="black",
    node_size=300,
    edge_color="#999",
)
plt.title(f"Every edge in the graph — plus {isolated} entities with no edges at all")
plt.axis("off")
plt.show()

**That is not a graph. It is a scatter.** A handful of edges across sixteen documents
means traversal has essentially nowhere to go — a one-hop walk from most entities
reaches nothing.

The reason is structural, not a tuning problem. This corpus is **sixteen independent
explanatory essays**. Each explains one topic thoroughly and barely mentions the others.
There is almost no entity co-occurrence to exploit, because the documents were not
written as an interconnected knowledge base.

### The thirty-second test: mean degree

*Degree* = how many neighbours an entity has. Below ~3 there is not enough structure to
traverse. This corpus scores about **1.6**. Building the graph and reading that one
number tells you more than any amount of tuning would.

## 4. Step 3 — retrieve by walking

In [ ]:
retriever = GraphRetriever(graph, hops=1)
for q in [
    "What is FUSION_CONSTANT set to?",
    "How does BM25 relate to HNSW?",
    "Why do I get charged twice?",
]:
    seeds = retriever._seed_entities(q)
    hits = retriever.retrieve(q, top_k=3)
    print(f"{q}\n   seed entities: {seeds or 'none — nothing to walk from'}")
    for h in hits:
        print(f"   {h.rank}. {h.chunk.doc_id:26s} graph={h.score:.2f}")

The last query names no entity, so the retriever correctly returns **nothing** rather
than inventing a starting point. A graph retriever that guesses a seed returns confident
noise.

## 5. Measure it against the golden set

In [ ]:
import pandas as pd

from ragkit.eval.golden import load_golden
from ragkit.eval.metrics import recall_at_k
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())

rows = []
for q in load_golden():
    g = list(dict.fromkeys(s.chunk.doc_id for s in retriever.retrieve(q.question, top_k=20)))
    r = list(dict.fromkeys(s.chunk.doc_id for s in pipeline.retrieve(q.question, top_k=20)))
    rows.append(
        {
            "kind": q.kind,
            "graph returned nothing": not g,
            "R@5 graph": recall_at_k(g, q.relevant, 5),
            "R@5 rerank": recall_at_k(r, q.relevant, 5),
        }
    )
table = pd.DataFrame(rows)
summary = table.groupby("kind").agg(
    n=("kind", "size"),
    empty=("graph returned nothing", "sum"),
    **{"R@5 graph": ("R@5 graph", "mean"), "R@5 rerank": ("R@5 rerank", "mean")},
)
summary.loc["all"] = [
    len(table),
    table["graph returned nothing"].sum(),
    table["R@5 graph"].mean(),
    table["R@5 rerank"].mean(),
]
summary.round(3)

Compare with the repo's run: graph R@5 **0.409** overall against **0.987** for rerank,
with the graph returning **nothing at all for 34 of 44 questions**.

`vocab_mismatch` scoring **0.000** is the clearest signal. Those questions are phrased in
a user's words — "Why do I get charged twice?" — so they contain no entity names *by
construction*. **A graph keyed on entities cannot answer a question that names no
entities**, which is precisely the case dense retrieval exists for. (The `unanswerable`
row scores 1.0 — an artifact: retrieving nothing is correct when there is nothing to
find.)

The honest deployment of GraphRAG is *alongside* vector retrieval for the minority of
questions that name entities and need their connections — never instead of it.

## 6. Two silent extraction bugs

Entity extraction is where GraphRAG quality is actually decided, it is lossy in ways
that never raise, and its junk is plausible enough to survive a casual look.

**Bug 1 — `\s` matches newlines.** The Title Case pattern originally used `\s` between
words, so it ran across a line break and swallowed the start of the next sentence:

In [ ]:
import re

text = "## Vector Search\nAn embedding model maps text to vectors."
buggy = re.compile(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3}\b")
fixed = re.compile(r"\b[A-Z][a-z]+(?:[ \t]+[A-Z][a-z]+){1,3}\b")
print("with \\s     :", buggy.findall(text))
print("with [ \\t]  :", fixed.findall(text))

**Bug 2 — singletons dropped entirely.** An entity seen in only one chunk cannot form an
edge, so it was dropped from the graph — which made `FUSION_CONSTANT` *unfindable* even
though the corpus documents it. Singletons are now excluded from edge-building but kept
as nodes, so a query naming one can still seed a walk:

In [ ]:
key = "fusion_constant"
print(
    "chunks containing it:",
    len(graph.entity_chunks.get(key, ())),
    "| neighbours:",
    graph.neighbours(key),
)
print(
    "still retrievable:",
    [h.chunk.doc_id for h in retriever.retrieve("What is FUSION_CONSTANT?", top_k=2)],
)

## 7. Build the version that *would* work

GraphRAG shines when **entities recur across documents in varying combinations**:
incident reports that reference the same services, papers citing each other, org
documents naming the same teams. Let's write five tiny incident reports with that
shape and build the same graph over them.

In [ ]:
from ragkit.types import Document

incidents = {
    "INC-101.md": "`AuthService` timed out because `SessionStore` was saturated. `PaymentsAPI` "
    "requests failed while `AuthService` was degraded. Owner: `TeamIdentity`.",
    "INC-102.md": "`SessionStore` failover caused stale reads in `CartService`. `TeamIdentity` "
    "rolled back the `SessionStore` config.",
    "INC-103.md": "`PaymentsAPI` retries amplified load on `LedgerDB`. `TeamPayments` added "
    "backoff; `CartService` checkout errors cleared.",
    "INC-104.md": "`LedgerDB` disk filled; `PaymentsAPI` and `ReportingJob` stalled. "
    "`TeamPayments` and `TeamData` resized volumes.",
    "INC-105.md": "`ReportingJob` overloaded `LedgerDB` replicas, slowing `PaymentsAPI`. "
    "`TeamData` moved the job off-peak.",
}
docs = [Document(doc_id=k, text=v, title=k) for k, v in incidents.items()]
small_chunks = chunk_documents(docs, RecursiveChunker(512, 0))
incident_graph = build_graph(small_chunks)
print("incident corpus:", incident_graph.stats())
print("course corpus:  ", graph.stats())

walker = GraphRetriever(incident_graph, hops=1)
q = "What else is affected when LedgerDB has problems?"
print(f"\n{q}\n   seeds: {walker._seed_entities(q)}")
for h in walker.retrieve(q, top_k=5):
    print(f"   {h.rank}. {h.chunk.doc_id}  graph={h.score:.2f}  {h.chunk.text[:70]}…")

In [ ]:
G2 = nx.Graph()
for a, neighbours in incident_graph.edges.items():
    for b, w in neighbours.items():
        G2.add_edge(incident_graph.display.get(a, a), incident_graph.display.get(b, b), weight=w)
plt.figure(figsize=(8, 4))
nx.draw_networkx(
    G2,
    nx.spring_layout(G2, seed=3),
    node_color="#55A868",
    font_size=9,
    node_size=400,
    edge_color="#999",
    width=[G2[u][v]["weight"] for u, v in G2.edges()],
)
plt.title("Five incident reports: a graph with somewhere to walk")
plt.axis("off")
plt.show()

Five short documents, and the mean degree is already far higher than sixteen essays.
"What else is affected when LedgerDB has problems?" walks to `PaymentsAPI`,
`ReportingJob` and the teams involved — a *connection* question that similarity search
would answer only partially.

## 8. Deciding whether you need this

Do not start from "should we add a knowledge graph". Start from your failing questions:

1. Take the questions your current pipeline gets wrong.
2. How many require **connecting information across documents** rather than finding
   the right passage?
3. If that number is small, GraphRAG will not help, however good the demos look.

Then build the graph and read the mean degree. **Reach for this last** — on most corpora
hybrid retrieval plus reranking answers the questions a graph would, at a fraction of
the build and maintenance cost.

## 9. Check your understanding

**1. Why did graph retrieval score 0.000 on vocabulary-mismatch questions?**

<details><summary>▶ Show answer</summary>

Those questions are phrased in everyday words and name no entities, so there is no
seed node to start a walk from. The retriever correctly returns nothing.

</details>

**2. What does a mean degree of 1.6 tell you, and is it fixable by tuning?**

<details><summary>▶ Show answer</summary>

Most entities connect to one or two others (or none), so traversal has nowhere to
go. It is a property of the corpus — independent essays with little entity
co-occurrence — not of the parameters.

</details>

**3. Why does `build_graph` prune hubs (`max_degree`)?**

<details><summary>▶ Show answer</summary>

An entity that co-occurs with almost everything connects everything to everything;
walking through it returns the whole corpus. Hubs are the graph equivalent of
stopwords.

</details>

## ✍️ Exercises

1. **Try to force a denser graph** by lowering `min_frequency` or adding patterns. Does
   mean degree rise meaningfully — and is the new structure real or junk?
2. **Break the extractor:** put `\s` back into the Title Case pattern (copy
   `ENTITY_PATTERNS`, pass your own `extractor` to `build_graph`) and list the junk
   entities that appear.
3. **Hybrid with a graph:** fuse `GraphRetriever` with the reranked pipeline using
   `reciprocal_rank_fusion` (lesson 04). Does it help the multi-hop questions at all?

## Key takeaways

1. Graphs answer connection questions that similarity search structurally cannot.
2. Co-occurrence graphs need entities that recur across documents in varying combinations.
3. Mean degree is a thirty-second test: this corpus scores ~1.6 — a scatter, not a graph.
4. A graph keyed on entities cannot answer questions that name none — use it alongside vectors.
5. Entity extraction is lossy and fails silently; inspect what it produces.

## Next

**[Lesson 12 — Production hardening and the inspector](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/12-production-hardening/colab.ipynb)**